In [ ]:
# 3. Jalankan sekali pada runtime baru (boleh dilewati bila versi berikut sudah terpasang).
# Jangan menghapus nomor versi: ini adalah baseline dependency untuk eksperimen.
%pip install --quiet \
    'transformers==4.46.3' \
    'datasets==3.1.0' \
    'accelerate==1.0.1' \
    'scikit-learn==1.5.2' \
    'pandas==2.2.3' \
    'numpy==1.26.4' \
    'evaluate==0.4.3' \
    'sentencepiece==0.2.0'

print('Pemasangan selesai. Jika ada pesan restart runtime, lakukan restart lalu ulangi notebook dari sel pertama.')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for sentencepiece (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for sentencepiece
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> sentencepiece
Pemasangan selesai. Jika ada pesan restart runtime, lakukan restart lalu ulangi notebook dari sel pertama.


# 00 — Lingkungan Colab dan Reproducibility

Notebook ini adalah titik awal untuk setiap eksperimen resmi Banking77. Jalankan dari atas ke bawah pada runtime Colab baru. Notebook tidak mengunduh, membersihkan, atau mengubah dataset mentah.

**Sebelum menjalankan BERT:** pilih *Runtime → Change runtime type → T4 GPU* (atau GPU yang tersedia), lalu catat hasil pemeriksaan perangkat di bawah. Untuk audit, SVM, dan benchmark latency, gunakan runtime CPU bila protokol mengharuskannya.

In [1]:
# 1. Mount Google Drive dan tentukan satu folder proyek.
# Ubah hanya jika nama/lokasi folder Anda di Drive berbeda.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Thesis_Banking77')
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'Folder proyek tidak ditemukan: {PROJECT_DIR}\n'
        'Periksa PROJECT_DIR; notebook ini sengaja tidak membuat folder proyek utama secara otomatis.'
    )

print(f'PROJECT_DIR: {PROJECT_DIR}')
print('Folder proyek ditemukan dan siap dipakai.')

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/Thesis_Banking77
Folder proyek ditemukan dan siap dipakai.


In [2]:
# 2. Buat struktur kerja standar TANPA menimpa berkas yang sudah ada.
PROJECT_SUBDIRS = [
    'notebooks',
    'data/raw',
    'data/processed',
    'data/splits',
    'configs',
    'runs',
    'reports',
    'app',
    'requirements',
]

for relative_path in PROJECT_SUBDIRS:
    (PROJECT_DIR / relative_path).mkdir(parents=True, exist_ok=True)

print('Subfolder siap:')
for relative_path in PROJECT_SUBDIRS:
    print(' -', PROJECT_DIR / relative_path)

Subfolder siap:
 - /content/drive/MyDrive/Thesis_Banking77/notebooks
 - /content/drive/MyDrive/Thesis_Banking77/data/raw
 - /content/drive/MyDrive/Thesis_Banking77/data/processed
 - /content/drive/MyDrive/Thesis_Banking77/data/splits
 - /content/drive/MyDrive/Thesis_Banking77/configs
 - /content/drive/MyDrive/Thesis_Banking77/runs
 - /content/drive/MyDrive/Thesis_Banking77/reports
 - /content/drive/MyDrive/Thesis_Banking77/app
 - /content/drive/MyDrive/Thesis_Banking77/requirements


## Dependensi

Sel ini memasang versi yang dicatat secara eksplisit untuk paket eksperimen utama. Setelah pemasangan selesai, **restart runtime** bila Colab meminta, lalu lanjutkan dari sel pertama. Versi PyTorch dan CUDA bawaan Colab tidak dipaksa diganti karena harus cocok dengan driver GPU runtime; keduanya tetap dicatat dalam artefak lingkungan.

In [3]:
# 3. Jalankan sekali pada runtime baru (boleh dilewati bila versi berikut sudah terpasang).
# Jangan menghapus nomor versi: ini adalah baseline dependency untuk eksperimen.
%pip install --quiet \
    'transformers==4.46.3' \
    'datasets==3.1.0' \
    'accelerate==1.0.1' \
    'scikit-learn==1.5.2' \
    'pandas==2.2.3' \
    'numpy==1.26.4' \
    'evaluate==0.4.3' \
    'sentencepiece==0.2.0'

print('Pemasangan selesai. Jika ada pesan restart runtime, lakukan restart lalu ulangi notebook dari sel pertama.')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for sentencepiece (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for sentencepiece
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> sentencepiece
Pemasangan selesai. Jika ada pesan restart runtime, lakukan restart lalu ulangi notebook dari sel pertama.


In [4]:
try:
    import sentencepiece
    print("✅ sentencepiece berhasil diimpor! Pemasangan aman untuk dilanjutkan.")
except ImportError as e:
    print("❌ sentencepiece GAGAL diimpor. Kita perlu memperbaikinya terlebih dahulu.")
    print(f"Error: {e}")

✅ sentencepiece berhasil diimpor! Pemasangan aman untuk dilanjutkan.


### Solusi jika GAGAL diimpor:
Jika hasil pengujian di atas menunjukkan kegagalan, Anda bisa mencoba salah satu solusi berikut:

1. **Pasang paket sistem prasyarat compiler** (karena `sentencepiece` membutuhkan kompilasi C++):
   ```bash
   !apt-get install -y cmake pkg-config libprotobuf-dev protobuf-compiler
   %pip install sentencepiece==0.2.0
   ```
2. **Gunakan versi yang sudah memiliki pre-compiled binary wheel** untuk arsitektur Anda (tanpa membatasi ke versi `0.2.0` jika diperbolehkan oleh baseline):
   ```bash
   %pip install sentencepiece
   ```

In [5]:
# 4. Konfigurasi run. Ubah PURPOSE, bukan RUN_ID, agar nama run tetap unik.
from datetime import datetime, timezone

EXPERIMENT_SEED = 42
PURPOSE = 'environment'  # contoh berikutnya: svm_validation atau bert_seed42
RUN_ID = f"{datetime.now().astimezone().strftime('%Y-%m-%d_%H%M%S')}_{PURPOSE}_seed{EXPERIMENT_SEED}"
RUN_DIR = PROJECT_DIR / 'runs' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)  # gagal jika ID kebetulan sudah ada; tidak menimpa run lain

for relative_path in ['metrics', 'predictions', 'models', 'logs']:
    (RUN_DIR / relative_path).mkdir(exist_ok=True)

print(f'RUN_ID : {RUN_ID}')
print(f'RUN_DIR: {RUN_DIR}')

RUN_ID : 2026-10-31_111441_environment_seed42
RUN_DIR: /content/drive/MyDrive/Thesis_Banking77/runs/2026-10-31_111441_environment_seed42


In [6]:
# 5. Tetapkan seed untuk Python, NumPy, dan PyTorch.
# Seed BERT final yang akan dipakai: 13, 42, dan 2026 — masing-masing sebagai run terpisah.
import os
import random
import numpy as np
import torch

def set_experiment_seed(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Deterministic meningkatkan keterulangan, tetapi dapat sedikit memperlambat training.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except AttributeError:
        pass

set_experiment_seed(EXPERIMENT_SEED)
print(f'Seed aktif: {EXPERIMENT_SEED}')

Seed aktif: 42


In [7]:
# 6. Kumpulkan bukti lingkungan. Tidak ada asumsi tentang jenis GPU Colab.
import json
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

def package_version(package_name: str):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None

def command_output(command):
    try:
        return subprocess.check_output(command, text=True, stderr=subprocess.STDOUT).strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as error:
        return f'not available: {error}'

gpu_available = torch.cuda.is_available()
gpu = {'available': gpu_available}
if gpu_available:
    properties = torch.cuda.get_device_properties(0)
    gpu.update({
        'name': torch.cuda.get_device_name(0),
        'memory_total_gib': round(properties.total_memory / 1024**3, 2),
        'device_count': torch.cuda.device_count(),
    })

environment = {
    'run_id': RUN_ID,
    'purpose': PURPOSE,
    'created_at_local': datetime.now().astimezone().isoformat(),
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'project_dir': str(PROJECT_DIR),
    'seed': EXPERIMENT_SEED,
    'python': sys.version,
    'platform': platform.platform(),
    'processor': platform.processor(),
    'cpu_count_logical': os.cpu_count(),
    'ram': command_output(['free', '-h']),
    'torch': torch.__version__,
    'torch_cuda_build': torch.version.cuda,
    'cudnn_version': torch.backends.cudnn.version(),
    'packages': {
        name: package_version(name)
        for name in ['transformers', 'datasets', 'accelerate', 'scikit-learn', 'pandas', 'numpy', 'evaluate', 'sentencepiece']
    },
    'gpu': gpu,
    'nvidia_smi': command_output(['nvidia-smi']),
}

print(json.dumps(environment, indent=2, ensure_ascii=False))

{
  "run_id": "2026-10-31_111441_environment_seed42",
  "purpose": "environment",
  "created_at_local": "2026-10-31T11:17:06.738297+00:00",
  "created_at_utc": "2026-10-31T11:17:06.738317+00:00",
  "project_dir": "/content/drive/MyDrive/Thesis_Banking77",
  "seed": 42,
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "processor": "x86_64",
  "cpu_count_logical": 2,
  "ram": "total        used        free      shared  buff/cache   available\nMem:            12Gi       1.7Gi       5.6Gi       5.8Mi       5.6Gi        10Gi\nSwap:             0B          0B          0B",
  "torch": "2.11.0+cu130",
  "torch_cuda_build": "13.0",
  "cudnn_version": 92700,
  "packages": {
    "transformers": "5.21.0",
    "datasets": "4.8.5",
    "accelerate": "1.15.0",
    "scikit-learn": "1.6.1",
    "pandas": "2.2.3",
    "numpy": "2.1.3",
    "evaluate": null,
    "sentencepiece": "0.2.2"
  },
  "gpu": {
    "available": true,
    "na

In [8]:
# 7. Simpan environment dan lock dependency ke Drive.
# File per-run tidak pernah menggantikan lock run sebelumnya.
environment_path = RUN_DIR / 'environment.json'
environment_path.write_text(json.dumps(environment, indent=2, ensure_ascii=False), encoding='utf-8')

pip_freeze = command_output([sys.executable, '-m', 'pip', 'freeze'])
run_lock_path = RUN_DIR / 'requirements.lock.txt'
run_lock_path.write_text(pip_freeze + '\n', encoding='utf-8')

# Manifest ringkas memudahkan pemasangan ulang paket inti pada runtime baru.
core_lock_lines = []
for name, installed_version in environment['packages'].items():
    if installed_version is not None:
        core_lock_lines.append(f'{name}=={installed_version}')
core_lock_path = PROJECT_DIR / 'requirements' / f'{RUN_ID}_core.lock.txt'
core_lock_path.write_text('\n'.join(core_lock_lines) + '\n', encoding='utf-8')

config = {
    'run_id': RUN_ID,
    'purpose': PURPOSE,
    'seed': EXPERIMENT_SEED,
    'environment_file': str(environment_path.relative_to(PROJECT_DIR)),
    'requirements_lock': str(run_lock_path.relative_to(PROJECT_DIR)),
    'core_requirements_lock': str(core_lock_path.relative_to(PROJECT_DIR)),
}
config_path = RUN_DIR / 'config.json'
config_path.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')

print('Tersimpan:')
for path in [environment_path, run_lock_path, core_lock_path, config_path]:
    print(' -', path)

Tersimpan:
 - /content/drive/MyDrive/Thesis_Banking77/runs/2026-10-31_111441_environment_seed42/environment.json
 - /content/drive/MyDrive/Thesis_Banking77/runs/2026-10-31_111441_environment_seed42/requirements.lock.txt
 - /content/drive/MyDrive/Thesis_Banking77/requirements/2026-10-31_111441_environment_seed42_core.lock.txt
 - /content/drive/MyDrive/Thesis_Banking77/runs/2026-10-31_111441_environment_seed42/config.json


## Selesai jika

- `runs/<RUN_ID>/environment.json`, `requirements.lock.txt`, dan `config.json` ada di Google Drive.
- GPU (atau ketiadaannya) tercatat apa adanya.
- Anda menjalankan eksperimen berikutnya dengan `RUN_ID` baru dan menyimpan konfigurasi, prediksi, metrik, serta artefaknya pada folder run tersebut.

Catatan: keterulangan numerik GPU tidak selalu bit-identik antarversi driver/Colab. Karena itu, versi perangkat lunak, CUDA, GPU, seed, dan hasil run harus selalu disimpan bersama metrik.